# TCInvest Raw Review Manual Inspection

This notebook is for **read-only human QA** of the current primary raw review datasets before downstream analysis. It supports inspection, searching, filtering, sorting, and sampling only. It does not modify raw or processed data, clean text, classify reviews, run sentiment/topic models, or select complaints.

In [21]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 300)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "01_data").is_dir() and (candidate / "03_notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the current working directory.")

PROJECT_ROOT = find_project_root()
APP_STORE_PATH = PROJECT_ROOT / "01_data/raw/app_store/2026-09-22_app_store_raw_expanded_v2.csv"
GOOGLE_PLAY_PATH = PROJECT_ROOT / "01_data/raw/google_play/2026-09-22_google_play_raw.csv"
CLEAN_PATH = PROJECT_ROOT / "01_data/processed/reviews_clean_full.csv"
REJECTED_PATH = PROJECT_ROOT / "01_data/processed/reviews_rejected.csv"

app_store_df = pd.read_csv(APP_STORE_PATH, dtype={"review_id": "string"})
google_play_df = pd.read_csv(GOOGLE_PLAY_PATH, dtype={"reviewId": "string"})

print(f"Project root: {PROJECT_ROOT}")
print(f"App Store input: {APP_STORE_PATH.relative_to(PROJECT_ROOT)}")
print(f"Google Play input: {GOOGLE_PLAY_PATH.relative_to(PROJECT_ROOT)}")

Project root: /Users/ndmaivy/Documents/Mai Vy/FBAS6/nhom-nam-nguoi
App Store input: 01_data/raw/app_store/2026-09-22_app_store_raw_expanded_v2.csv
Google Play input: 01_data/raw/google_play/2026-09-22_google_play_raw.csv


## Dataset Overview

In [22]:
app_dates = pd.to_datetime(app_store_df["updated_at"], utc=True, errors="coerce")
google_dates = pd.to_datetime(google_play_df["at"], utc=True, errors="coerce")

assert len(app_store_df) == 522
assert app_store_df["review_id"].nunique() == 522
assert app_dates.min() == pd.Timestamp("2023-04-14T03:22:33+00:00")
assert app_dates.max() == pd.Timestamp("2026-09-14T07:44:28+00:00")
assert len(google_play_df) == 3223
assert google_play_df["reviewId"].nunique() == 3223
assert google_dates.min() == pd.Timestamp("2015-08-13T11:10:30+00:00")
assert google_dates.max() == pd.Timestamp("2026-09-21T12:13:27+00:00")

overview = pd.DataFrame([
    {"source": "App Store", "rows": len(app_store_df), "unique_review_ids": app_store_df["review_id"].nunique(), "earliest": app_dates.min(), "latest": app_dates.max()},
    {"source": "Google Play", "rows": len(google_play_df), "unique_review_ids": google_play_df["reviewId"].nunique(), "earliest": google_dates.min(), "latest": google_dates.max()},
])
display(overview)
print(f"Total current primary raw records: {len(app_store_df) + len(google_play_df):,}")

for label, frame in [("APP STORE", app_store_df), ("GOOGLE PLAY", google_play_df)]:
    print(f"\n{label}: shape={frame.shape}")
    print("Columns:", frame.columns.tolist())
    display(frame.dtypes.rename("dtype").to_frame())
    display(frame.head(3))

,source,rows,unique_review_ids,earliest,latest
0,App Store,522,522,2023-04-14 03:22:33+00:00,2026-09-14 07:44:28+00:00
1,Google Play,3223,3223,2015-08-13 11:10:30+00:00,2026-09-21 12:13:27+00:00


Total current primary raw records: 3,745

APP STORE: shape=(522, 19)
Columns: ['review_id', 'user_name', 'rating', 'title', 'content', 'app_version', 'updated_at', 'vote_sum', 'vote_count', 'source', 'endpoint', 'lang', 'storefront', 'raw_occurrence_count', 'raw_source_files', 'source_groups', 'endpoint_groups', 'belongs_to_tcinvest_response', 'substantive_conflict_fields']


,dtype
review_id,string
user_name,str
rating,int64
title,str
content,str
app_version,str
updated_at,str
vote_sum,int64
vote_count,int64
source,str


,review_id,user_name,rating,title,content,app_version,updated_at,vote_sum,vote_count,source,endpoint,lang,storefront,raw_occurrence_count,raw_source_files,source_groups,endpoint_groups,belongs_to_tcinvest_response,substantive_conflict_fields
0,14548034358,Lethuylinh1004,1,Liên tục lỗi không thể truy cập,"Là một người dùng lâu năm của tech, ban đầu rất hài lòng về chất lượng và giao diện của app, dễ giao dịch. Nhưng vài tháng trở lại đây app lỗi rất nhiều, chỉ đăng nhập có lúc đợi cả 5p vẫn chưa đăng nhập vào trang chủ được, thứ 2 là quảng cáo nhiều, truy cập vào được tài khoản thì phải tắt 7749 ...",6.4.6,2026-09-14T00:44:28-07:00,0,0,App Store,vn|json|default|page=1,unknown,vn,5,diagnostics/recency_2026-09-22/vn_page_01_default.json;diagnostics/recency_2026-09-22/vn_page_01_mostHelpful.xml;diagnostics/recency_2026-09-22/vn_top_mostHelpful.json;diagnostics/recency_2026-09-22/vn_top_mostHelpful.xml;diagnostics/recency_2026-09-22/vn_top_mostrecent.json,diagnostic_recency_2026-09-22,vn|json|default|page=1;vn|json|mostHelpful|page=top;vn|json|mostrecent|page=top;vn|xml|mostHelpful|page=1;vn|xml|mostHelpful|page=top,True,NaN
1,14520064997,Abcdef847362,1,Tần suất bị lỗi mục tài sản quá nhiều,Rất hay bị lỗi báo k có tiền k có cổ phiếu nào trong tài khoản,6.4.6,2026-09-06T21:11:54-07:00,0,0,App Store,vn|json|default|page=1,unknown,vn,5,diagnostics/recency_2026-09-22/vn_page_01_default.json;diagnostics/recency_2026-09-22/vn_page_01_mostHelpful.xml;diagnostics/recency_2026-09-22/vn_top_mostHelpful.json;diagnostics/recency_2026-09-22/vn_top_mostHelpful.xml;diagnostics/recency_2026-09-22/vn_top_mostrecent.json,diagnostic_recency_2026-09-22,vn|json|default|page=1;vn|json|mostHelpful|page=top;vn|json|mostrecent|page=top;vn|xml|mostHelpful|page=1;vn|xml|mostHelpful|page=top,True,NaN
2,14520054584,Vy11229,2,Khá tệ,"App rất lag, đôi khi còn không vào được",6.4.6,2026-09-06T21:07:12-07:00,0,0,App Store,vn|json|default|page=1,unknown,vn,5,diagnostics/recency_2026-09-22/vn_page_01_default.json;diagnostics/recency_2026-09-22/vn_page_01_mostHelpful.xml;diagnostics/recency_2026-09-22/vn_top_mostHelpful.json;diagnostics/recency_2026-09-22/vn_top_mostHelpful.xml;diagnostics/recency_2026-09-22/vn_top_mostrecent.json,diagnostic_recency_2026-09-22,vn|json|default|page=1;vn|json|mostHelpful|page=top;vn|json|mostrecent|page=top;vn|xml|mostHelpful|page=1;vn|xml|mostHelpful|page=top,True,NaN



GOOGLE PLAY: shape=(3223, 16)
Columns: ['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion', 'source', 'collection_language', 'collection_country', 'collection_locale', 'retrieved_from_locales']


,dtype
reviewId,string
userName,str
userImage,str
content,str
score,int64
thumbsUpCount,int64
reviewCreatedVersion,str
at,str
replyContent,str
repliedAt,str


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,source,collection_language,collection_country,collection_locale,retrieved_from_locales
0,6638c299-2864-4386-837e-41064b75be48,Thanh Son Huynh,https://play-lh.googleusercontent.com/a-/ALV-UjWVYyexHJs75plCofmmf_iI7Pk1lf4Io7Z1kkaN4-AfMoEUGsE,sao cứ cứng đơ không nhích nhút gõ bất kỳ một chữ nào được....,1,0,NaN,2026-09-21 12:13:27,NaN,NaN,NaN,Google Play,vi,vn,vi_vn,vi_vn
1,45b7bb0f-cd74-43a9-9b85-bd050b6a1f9f,Trần Sáng,https://play-lh.googleusercontent.com/a-/ALV-UjWDFhUdypUfx0qgj3S14QrZKIX8zk_nR6cCNL_RERIgIa6itHms,Mới dùng thử . App giao diên quá chán. Dùng tcbs . Mà phải vào ssi xem giá . .ap quá chán luôn .,1,0,6.4.5,2026-09-21 05:51:18,NaN,NaN,6.4.5,Google Play,vi,vn,vi_vn,vi_vn
2,4ae66065-e785-4516-a20d-51040c7a80d2,Đạt Triệu Phú,https://play-lh.googleusercontent.com/a-/ALV-UjUAi2T5ImLrtRxmPZRNM-XbtRV83EVDxssew8Cwt3ZXKg1u6dE,"rẻ rách, đã rườm rà còn khó xài",1,0,6.4.5,2026-09-18 18:49:22,NaN,NaN,6.4.5,Google Play,vi,vn,vi_vn,vi_vn


## Readable Preview Tables

The App Store `lang` field is shown only as saved metadata; it may be unknown or heuristic and is not treated as language ground truth. Google Play `collection_locale` is collection metadata, not detected review language.

In [23]:
app_preview_columns = ["review_id", "rating", "title", "content", "updated_at", "app_version", "storefront"]
if "lang" in app_store_df.columns:
    app_preview_columns.append("lang")
app_store_preview = app_store_df[app_preview_columns].copy()

google_preview_columns = ["reviewId", "score", "content", "at", "reviewCreatedVersion", "appVersion", "collection_locale"]
google_play_preview = google_play_df[google_preview_columns].copy()

print("App Store readable preview")
display(app_store_preview.head(10))
print("Google Play readable preview")
display(google_play_preview.head(10))

App Store readable preview


,review_id,rating,title,content,updated_at,app_version,storefront,lang
0,14548034358,1,Liên tục lỗi không thể truy cập,"Là một người dùng lâu năm của tech, ban đầu rất hài lòng về chất lượng và giao diện của app, dễ giao dịch. Nhưng vài tháng trở lại đây app lỗi rất nhiều, chỉ đăng nhập có lúc đợi cả 5p vẫn chưa đăng nhập vào trang chủ được, thứ 2 là quảng cáo nhiều, truy cập vào được tài khoản thì phải tắt 7749 ...",2026-09-14T00:44:28-07:00,6.4.6,vn,unknown
1,14520064997,1,Tần suất bị lỗi mục tài sản quá nhiều,Rất hay bị lỗi báo k có tiền k có cổ phiếu nào trong tài khoản,2026-09-06T21:11:54-07:00,6.4.6,vn,unknown
2,14520054584,2,Khá tệ,"App rất lag, đôi khi còn không vào được",2026-09-06T21:07:12-07:00,6.4.6,vn,unknown
3,14504211053,3,Quá chậm,"Áp giao dịch gì mà chậm như rùa,mong bqt cải tiến",2026-09-02T20:32:43-07:00,6.4.6,vn,unknown
4,14468902446,1,Nghi vấn bán dữ liệu cho lái,"Tôi cầm cổ phiếu DBC 4 tháng, ngay khoảnh khắc khớp lệnh mua, cổ phiếu không một lần giá xanh, liên tiếp giảm 4 tháng liền. Tôi chấp nhận nạp tiền giữ vị thế margin, tuy nhiên sau 4 tháng, tôi muốn đổi cổ phiếu để có vị thế khác, ngay khoảnh khắc bán thì cổ phiếu DBC khớp lệnh và bay liên tiếp. ...",2026-08-24T21:00:51-07:00,6.4.4,vn,unknown
5,14467591535,5,NGUYEN VAN PHUONG,Namkuto,2026-08-24T12:29:06-07:00,6.4.4,vn,unknown
6,14465859621,1,"Giao diện rối khó sử dụng, CSKH kém, không thể gọi được NV CSKH","- UI UX kém, dễ bị bấm nhầm mua và bán\n- hotline chỉ để làm cảnh, AI trả lời tự động không giải quyết được vấn đề\n- không gọi được nv cskh, email không trả lời",2026-08-24T03:08:22-07:00,6.4.4,vn,unknown
7,14465018475,1,App lỗi quá nhiều,Nói chung quá tệ,2026-08-23T20:57:52-07:00,6.4.4,vn,unknown
8,14445501649,1,UI UX lởm quá,"UI UX này hoàn toàn ko phù hợp với iPad, chắc dev lười nên bê nguyên web sang. App chạy rất bất tiện, thao tác hoàn toàn lệch, ko hề chính xác. Dùng app thế này mà giao dịch thì lỗ nặng.",2026-08-18T22:09:21-07:00,6.4.4,vn,unknown
9,14439296592,1,Cách xác thực sự quá ngu,Lần đầu tiên tôi thấy việc xác thực để mở connection nhận data từ websocket nó ngớ ngẩn như vậy. Cứ 24 tiếng lại phải chạy vô app lấy cái OTP bằng tay xong đi cập nhật biến môi trường trước giờ giao dịch để k bị miss data. Đều đặn hằng ngày. Dev bên này nên thử mở cái docs của DNSE với SSi họ là...,2026-08-17T08:52:20-07:00,6.4.4,vn,unknown


Google Play readable preview


,reviewId,score,content,at,reviewCreatedVersion,appVersion,collection_locale
0,6638c299-2864-4386-837e-41064b75be48,1,sao cứ cứng đơ không nhích nhút gõ bất kỳ một chữ nào được....,2026-09-21 12:13:27,NaN,NaN,vi_vn
1,45b7bb0f-cd74-43a9-9b85-bd050b6a1f9f,1,Mới dùng thử . App giao diên quá chán. Dùng tcbs . Mà phải vào ssi xem giá . .ap quá chán luôn .,2026-09-21 05:51:18,6.4.5,6.4.5,vi_vn
2,4ae66065-e785-4516-a20d-51040c7a80d2,1,"rẻ rách, đã rườm rà còn khó xài",2026-09-18 18:49:22,6.4.5,6.4.5,vi_vn
3,f93f9a8e-c094-4ceb-8268-f4ba73dbb18c,1,"bắt làm chữ ký số mà cứ chụp ảnh xong là đơ, ko làm đc cái gì nữa, xóa đi cài lại vẫn thế",2026-09-15 22:04:51,6.4.5,6.4.5,vi_vn
4,ef9573f5-cac7-4b27-857a-f29bebf2c4d2,3,"Tôi đang sở hữu quỹ tcef đặt mua thêm ccq ngay trong tcef mà khi đặt lệnh lại lòi ra lệnh mua tcbf là sao vậy, ai code app kỳ vậy",2026-09-14 08:56:29,6.4.5,6.4.5,vi_vn
5,b60631dc-52f7-4a6e-ab62-777981d617ec,1,"Giao diện app quá tệ. Tính năng cơ bản như xem chart và đặt lệnh chuyển qua lại rất phức tạp, tôi muốn vừa xem chart nhanh vừa đặt lệnh nhiều ngày (24/7) đơn giản, mà app quá rườm rà, giao diện nhiều tính năng rác không dùng đến chiếm hết chỗ. Muốn xem danh mục của tôi đang giữ biến động thế nào...",2026-09-11 15:57:15,6.4.5,6.4.5,vi_vn
6,7d25eb7f-ac40-4de4-8943-5ab02a03cdbf,1,"App Techcombank Mobile nhanh bao nhiêu thì app TCBS chậm bấy nhiêu, đó là vì TCB mobile có giao diện đơn giản. Để cải thiện, các bạn cần đơn giản hóa giao diện, các cửa sổ lựa chọn chỉ nên hiện những gì cần thiết, cơ bản (các tính năng thông minh khác dành cho nhà đầu tư chuyên nghiệp nên được c...",2026-09-09 06:36:39,NaN,NaN,vi_vn
7,84a4e782-1615-4764-ad24-3974c5da8eb0,2,app lag giật khó dùng,2026-09-08 08:36:53,6.4.5,6.4.5,vi_vn
8,74c2b3de-07a9-4e83-afe4-76ce8134f8f0,1,"Rất tệ yêu cầu mã iOTP mà chờ gần 30p không có , liên hệ nhân viên thì không được , hệ thống quản lý quá kém",2026-08-29 21:30:32,6.4.5,6.4.5,vi_vn
9,5bac4bc7-ad1e-45fe-858c-84c7185e971a,1,"Ứng dụng lỏi, vô tạo tài khoản bấm tiếp tục mãi k đc",2026-08-28 23:10:48,6.4.14,6.4.14,vi_vn


## Random Review Sample

In [24]:
display(app_store_preview.sample(n=min(20, len(app_store_preview)), random_state=42))
display(google_play_preview.sample(n=min(20, len(google_play_preview)), random_state=42))

,review_id,rating,title,content,updated_at,app_version,storefront,lang
434,11737328061,5,Tuyệt vời,105CA18669,2024-09-18T05:47:47-07:00,5.7,vn,unknown
440,11737258935,5,Dễ sử dụng,"Giao diện app dễ sử dụng, đẹp",2024-09-18T05:26:06-07:00,5.7,vn,unknown
6,14465859621,1,"Giao diện rối khó sử dụng, CSKH kém, không thể gọi được NV CSKH","- UI UX kém, dễ bị bấm nhầm mua và bán\n- hotline chỉ để làm cảnh, AI trả lời tự động không giải quyết được vấn đề\n- không gọi được nv cskh, email không trả lời",2026-08-24T03:08:22-07:00,6.4.4,vn,unknown
184,13014069107,1,lỗi chuyển tiền,k rút tiền dc,2025-08-14T01:51:40-07:00,6.0.2,vn,unknown
78,13835758727,1,TK ng thân không nộp được tiền,"Mình tạo tk con, Lần 1 nộp bằng cú pháp khi nhấn nộp tiền trên áp, 2 ngày sau trả lại báo sai cú pháp, trong khi cú pháp tự đông gen ra\n\nLần này 4 ngày rồi không back lại tiền, mà back lại cũng k biết làm thế nào để nộp",2026-03-10T21:52:51-07:00,6.4.2,vn,unknown
299,12092046546,2,Phần đặt lệnh cổ phiếu thực sự chán. Đơn điệu không hiển được thông tin thị trường thời điểm đó.,"Nhiều lúc đặt giá xong mở ra xem giá tt đã thay đổi từ lúc nào. Phần đặt khối lượng cũng vậy, nhiều lúc quên ko biết khối lượng cp của mình có bao nhiêu lại phải quay lại từ đầu để xem. Thực sự rất khó chịu",2024-12-22T01:52:10-07:00,5.8.2,vn,unknown
521,9819441432,3,Hay bị lỗi app,Cứ vài ngày là lỗi app phãi xoá đi tải lại hoài.,2023-04-13T20:22:33-07:00,5.0.1,vn,unknown
484,11736926631,5,Dich vu tot,105CH90456,2024-09-18T03:24:09-07:00,5.7,vn,unknown
117,13630948771,1,Lỗi ai chịu,"Đang lỗi không giao dịch được, hãy sử lý và có lời giải thích thoả đáng. Lỗi hơn 1 tiếng chưa có trả lời khách hàng",2026-01-13T19:07:52-07:00,6.3,vn,unknown
137,13500073382,2,Biểu đồ kỹ thuật,Biểu đồ kỹ thuật chỉ cho xem theo hình dọc hoặc quay ngang thì không được full màn xem rất khó chịu. TCBS khắc phục thêm quả này nữa thì ngon,2025-12-10T22:23:18-07:00,6.2,vn,unknown


,reviewId,score,content,at,reviewCreatedVersion,appVersion,collection_locale
299,abff860a-3329-408e-a188-cf64dac8a665,1,"Ứng dụng rối nuồi khó sử dụng, lag, lỗi",2025-01-01 16:09:11,5.8.11,5.8.11,vi_vn
501,5db4f5b8-b014-46c1-ade1-9220e003ca81,1,Rối rắm. Khó dùng. Lag. Chậm. Nếu có 0 sao thì 1 sao cũng k xứng đáng,2024-08-02 09:27:56,NaN,NaN,vi_vn
631,08de7a4a-fecb-4600-a727-064057aee775,5,105C0910TA tôi thích bảng giá,2024-03-04 21:45:44,5.4.1,5.4.1,vi_vn
2098,9229dec5-3fce-4985-b83a-8b5da1fb7817,2,Lúc nào cũng gặp trục trặc với iOTP,2021-08-25 23:26:43,3.1.1,3.1.1,vi_vn
1377,b840a5c7-9bbb-4e6f-badd-e6c1fd2f5d4a,1,Cài hoài ko được,2022-04-26 19:56:17,NaN,NaN,vi_vn
1709,cc027484-9281-42c5-a44f-5bb297451ded,1,Ức chế những lúc thị trường điều chỉnh mạnh. Thích là không cho đặt lệnh. Buồn buồn là chơi lớn cấm cửa không cho giao dịch. Quá tệ.,2021-11-23 11:41:12,3.1.1,3.1.1,vi_vn
564,b9dac638-cc17-4d03-a456-ccf173c9464d,3,"Xoay ngang chưa được tốt lắm, dọc thì full màn, cần bổ sung thêm chỉ báo vào biểu đồ để có thể thuận tiện phân tích. Đa phần ở mức cơ bản",2024-04-03 00:17:36,5.5.2,5.5.2,vi_vn
1323,ed66daf9-cc37-4ec8-a2f0-a95c6580beb1,1,"App cực ngu, tìm kiếm, sao kê tài khoản, thực hiện margin tìm tất cả danh mục không thấy , sắp xếp rối tít mù . Mặc dù cũng dân công nghệ mà ko dùng nổi cái app này",2022-07-18 19:26:33,3.3.2,3.3.2,vi_vn
3060,d1e182cd-53a7-4a66-b8b1-de6afda48329,2,Please update app tobe able modify/cancel sell/buy command base on stock exchange. It should not lock all command on other stock like HNX UpCom,2021-06-10 15:01:54,3.0,3.0,en_vn
438,166a3e51-08f2-444f-adc9-edf731626ca0,5,Đâu tư tài chính,2024-08-16 07:59:48,5.7,5.7,vi_vn


## Review by Rating

Change either variable to 1, 2, 3, 4, or 5. A rating is displayed as metadata and is not automatically interpreted as sentiment.

In [25]:
APP_STORE_RATING = 1
GOOGLE_PLAY_RATING = 1

app_rating_matches = app_store_preview[pd.to_numeric(app_store_preview["rating"], errors="coerce").eq(APP_STORE_RATING)]
google_rating_matches = google_play_preview[pd.to_numeric(google_play_preview["score"], errors="coerce").eq(GOOGLE_PLAY_RATING)]
print(f"App Store matches: {len(app_rating_matches):,}")
display(app_rating_matches.head(30))
print(f"Google Play matches: {len(google_rating_matches):,}")
display(google_rating_matches.head(30))

App Store matches: 217


,review_id,rating,title,content,updated_at,app_version,storefront,lang
0,14548034358,1,Liên tục lỗi không thể truy cập,"Là một người dùng lâu năm của tech, ban đầu rất hài lòng về chất lượng và giao diện của app, dễ giao dịch. Nhưng vài tháng trở lại đây app lỗi rất nhiều, chỉ đăng nhập có lúc đợi cả 5p vẫn chưa đăng nhập vào trang chủ được, thứ 2 là quảng cáo nhiều, truy cập vào được tài khoản thì phải tắt 7749 ...",2026-09-14T00:44:28-07:00,6.4.6,vn,unknown
1,14520064997,1,Tần suất bị lỗi mục tài sản quá nhiều,Rất hay bị lỗi báo k có tiền k có cổ phiếu nào trong tài khoản,2026-09-06T21:11:54-07:00,6.4.6,vn,unknown
4,14468902446,1,Nghi vấn bán dữ liệu cho lái,"Tôi cầm cổ phiếu DBC 4 tháng, ngay khoảnh khắc khớp lệnh mua, cổ phiếu không một lần giá xanh, liên tiếp giảm 4 tháng liền. Tôi chấp nhận nạp tiền giữ vị thế margin, tuy nhiên sau 4 tháng, tôi muốn đổi cổ phiếu để có vị thế khác, ngay khoảnh khắc bán thì cổ phiếu DBC khớp lệnh và bay liên tiếp. ...",2026-08-24T21:00:51-07:00,6.4.4,vn,unknown
6,14465859621,1,"Giao diện rối khó sử dụng, CSKH kém, không thể gọi được NV CSKH","- UI UX kém, dễ bị bấm nhầm mua và bán\n- hotline chỉ để làm cảnh, AI trả lời tự động không giải quyết được vấn đề\n- không gọi được nv cskh, email không trả lời",2026-08-24T03:08:22-07:00,6.4.4,vn,unknown
7,14465018475,1,App lỗi quá nhiều,Nói chung quá tệ,2026-08-23T20:57:52-07:00,6.4.4,vn,unknown
8,14445501649,1,UI UX lởm quá,"UI UX này hoàn toàn ko phù hợp với iPad, chắc dev lười nên bê nguyên web sang. App chạy rất bất tiện, thao tác hoàn toàn lệch, ko hề chính xác. Dùng app thế này mà giao dịch thì lỗ nặng.",2026-08-18T22:09:21-07:00,6.4.4,vn,unknown
9,14439296592,1,Cách xác thực sự quá ngu,Lần đầu tiên tôi thấy việc xác thực để mở connection nhận data từ websocket nó ngớ ngẩn như vậy. Cứ 24 tiếng lại phải chạy vô app lấy cái OTP bằng tay xong đi cập nhật biến môi trường trước giờ giao dịch để k bị miss data. Đều đặn hằng ngày. Dev bên này nên thử mở cái docs của DNSE với SSi họ là...,2026-08-17T08:52:20-07:00,6.4.4,vn,unknown
10,14425846977,1,"Làm ăn ko kỹ càng, cẩu thả","Các công ty đã rời sàn như HLG thì bị tcinvest loại luôn khỏi list, làm nhà đầu tư mất luôn thông tin số lượng cổ phiếu đang sở hữu. Ko thể nào biết để gặp cty HLG làm giấy xác nhận. Nhiều mã rời sàn như hiện nay thì mất rất nhiều. List tôi gồm 100 loại cổ phiếu nên ko nhớ được",2026-08-14T01:14:42-07:00,6.4.4,vn,unknown
12,14418152911,1,Quá phiền hà,"Phục vụ quá cứng nhắc thay số điện thoại, cccd bắt yêu cầu mở thẻ ngân hàng để xác minh thông tin.",2026-08-12T03:46:04-07:00,6.4.4,vn,unknown
13,14414751396,1,App như cặc,App như lol mỗi vào để điền thông tin xác thực thôi mà đen xì ko hiện ra 1 cái gì . Ngân hàng lớn mà thuê team IT trình độ kém quá,2026-08-11T07:10:45-07:00,6.4.4,vn,unknown


Google Play matches: 1,799


,reviewId,score,content,at,reviewCreatedVersion,appVersion,collection_locale
0,6638c299-2864-4386-837e-41064b75be48,1,sao cứ cứng đơ không nhích nhút gõ bất kỳ một chữ nào được....,2026-09-21 12:13:27,NaN,NaN,vi_vn
1,45b7bb0f-cd74-43a9-9b85-bd050b6a1f9f,1,Mới dùng thử . App giao diên quá chán. Dùng tcbs . Mà phải vào ssi xem giá . .ap quá chán luôn .,2026-09-21 05:51:18,6.4.5,6.4.5,vi_vn
2,4ae66065-e785-4516-a20d-51040c7a80d2,1,"rẻ rách, đã rườm rà còn khó xài",2026-09-18 18:49:22,6.4.5,6.4.5,vi_vn
3,f93f9a8e-c094-4ceb-8268-f4ba73dbb18c,1,"bắt làm chữ ký số mà cứ chụp ảnh xong là đơ, ko làm đc cái gì nữa, xóa đi cài lại vẫn thế",2026-09-15 22:04:51,6.4.5,6.4.5,vi_vn
5,b60631dc-52f7-4a6e-ab62-777981d617ec,1,"Giao diện app quá tệ. Tính năng cơ bản như xem chart và đặt lệnh chuyển qua lại rất phức tạp, tôi muốn vừa xem chart nhanh vừa đặt lệnh nhiều ngày (24/7) đơn giản, mà app quá rườm rà, giao diện nhiều tính năng rác không dùng đến chiếm hết chỗ. Muốn xem danh mục của tôi đang giữ biến động thế nào...",2026-09-11 15:57:15,6.4.5,6.4.5,vi_vn
6,7d25eb7f-ac40-4de4-8943-5ab02a03cdbf,1,"App Techcombank Mobile nhanh bao nhiêu thì app TCBS chậm bấy nhiêu, đó là vì TCB mobile có giao diện đơn giản. Để cải thiện, các bạn cần đơn giản hóa giao diện, các cửa sổ lựa chọn chỉ nên hiện những gì cần thiết, cơ bản (các tính năng thông minh khác dành cho nhà đầu tư chuyên nghiệp nên được c...",2026-09-09 06:36:39,NaN,NaN,vi_vn
8,74c2b3de-07a9-4e83-afe4-76ce8134f8f0,1,"Rất tệ yêu cầu mã iOTP mà chờ gần 30p không có , liên hệ nhân viên thì không được , hệ thống quản lý quá kém",2026-08-29 21:30:32,6.4.5,6.4.5,vi_vn
9,5bac4bc7-ad1e-45fe-858c-84c7185e971a,1,"Ứng dụng lỏi, vô tạo tài khoản bấm tiếp tục mãi k đc",2026-08-28 23:10:48,6.4.14,6.4.14,vi_vn
10,9450f8ac-d9ba-4f47-b7f2-3271066a7707,1,app lỗi vcl. không quay vid xác thực khuôn mặt được. quá tệ,2026-08-25 10:25:19,6.4.5,6.4.5,vi_vn
13,b80c2d64-7ce6-4036-9237-860697ee5178,1,"chậm, lag quá ạ",2026-08-12 15:23:48,6.4.5,6.4.5,vi_vn


## Keyword Search

Examples: `đăng nhập`, `otp`, `chậm`, `lag`, `rút tiền`, `mua`, `bán`, `giao diện`, `phí`, `hỗ trợ`. Search results are not classifications.

In [26]:
KEYWORD = "đăng nhập"

app_search_text = app_store_df["title"].fillna("").astype(str) + " " + app_store_df["content"].fillna("").astype(str)
app_keyword_matches = app_store_preview[app_search_text.str.contains(KEYWORD, case=False, na=False, regex=False)]
google_keyword_matches = google_play_preview[google_play_df["content"].str.contains(KEYWORD, case=False, na=False, regex=False)]
print(f"App Store keyword matches for {KEYWORD!r}: {len(app_keyword_matches):,}")
display(app_keyword_matches.head(50))
print(f"Google Play keyword matches for {KEYWORD!r}: {len(google_keyword_matches):,}")
display(google_keyword_matches.head(50))

App Store keyword matches for 'đăng nhập': 14


,review_id,rating,title,content,updated_at,app_version,storefront,lang
0,14548034358,1,Liên tục lỗi không thể truy cập,"Là một người dùng lâu năm của tech, ban đầu rất hài lòng về chất lượng và giao diện của app, dễ giao dịch. Nhưng vài tháng trở lại đây app lỗi rất nhiều, chỉ đăng nhập có lúc đợi cả 5p vẫn chưa đăng nhập vào trang chủ được, thứ 2 là quảng cáo nhiều, truy cập vào được tài khoản thì phải tắt 7749 ...",2026-09-14T00:44:28-07:00,6.4.6,vn,unknown
15,14381104681,1,"Ko rút tiền được, bị giam tiền","Ko đăng nhập được để rút tiền, lúc cần thì ko rút tiền được phải đi vay tiền nóng.",2026-08-02T20:15:17-07:00,6.4.4,vn,unknown
16,14354074084,1,Bảo mật ngou như con bò,Dcmm xoá đi tải lại thì bắt xác nhận đăng nhập iotp trên máy đã đăng kí trc đó trong khi dùng đúng máy đó nhưng lại k cho đăng nhập vào dcm code ngoulol,2026-07-27T05:11:15-07:00,6.4.4,vn,unknown
21,14296658625,1,App rất thường bị lỗi,Rất thường gặp lỗi đăng nhập,2026-07-12T21:19:17-07:00,6.4.4,vn,unknown
59,13987803897,1,Đăng nhập lỗi,Đăng nhập toàn quay. Không vào được app,2026-04-23T01:17:00-07:00,6.4.3,vn,unknown
68,13884487026,1,Lỗi,Widget bị lỗi phiên đăng nhập đã hết hạn dù đã đăng nhập rồi. Thường xuyên bị lỗi vừa mở app đã bị dừng đột ngột,2026-03-24T23:45:50-07:00,6.4.3,vn,unknown
99,13714566655,1,Cá con,App chán toàn đơ lag lúc cao điểm giờ thêm bảo trì vào thời gian gd. Giam chân nđt.\nMua điện thoại mới đăng nhập không được mặc dù có số tk và mật khẩu và Sdt chính chủ rồi. Bắt xác nhận điện thoại cũ (đã cho và đã cài lại máy và người nhận cũng đã đi xa) rất bất tiện. Vậy trường hợp mất máy th...,2026-02-04T22:19:24-07:00,6.4.1,vn,unknown
127,13628373156,1,Giao diện xấu,"App giao diện xấu, rất khó hiểu, quy trình đăng nhập rườm rà gây trải nghiệm người dùng ko tốt. Nên thuê UX designer xây lại app giống app Techcombank",2026-01-13T03:50:01-07:00,6.3,vn,unknown
176,13068775356,1,Còn cần cải thiện,"Đăng nhập trên XS Max lại hiện option vân tay, nghĩ nó chán",2025-08-27T19:14:25-07:00,6.0.2,vn,unknown
279,12387003343,1,Lỗi đăng nhập - gọi hỗ trợ chỉ có tổng đài tự động,Hỗ trợ kém,2025-03-05T19:00:38-07:00,5.8.5,vn,unknown


Google Play keyword matches for 'đăng nhập': 82


,reviewId,score,content,at,reviewCreatedVersion,appVersion,collection_locale
44,c94d9d46-d3f6-4d2b-a423-99fd466da2bf,1,"ứng dụng lag lỏ , lỗi ngay từ đăng nhập , lỗi vân tay đen màn hình",2026-06-05 09:00:28,6.4.5,6.4.5,vi_vn
66,9affbb8e-9af2-4acd-945b-ddcac4ea8907,1,"2 hôm nay ko thể vào đc, cứ bị văng ra ko cho đăng nhập, quá tệ",2026-04-17 10:07:15,6.4.1,6.4.1,vi_vn
72,8601f3eb-e66c-468b-a6cc-f536335533e3,1,16.04.2026 đăng nhập liên tục mà không vào được gọi điện thoạ gần 10 cuộc cũng không được nhắn tin thì cho con AI ngoo ngốc trả lời. khó hiểu thật sự,2026-04-16 12:13:29,6.4.1,6.4.1,vi_vn
73,c5d09130-930e-454b-9af7-fda1b6a1da19,2,lỗi ko đăng nhập dc.,2026-04-16 10:11:05,6.4.1,6.4.1,vi_vn
75,ade96f2c-c95d-4486-8c9b-17394d3bd7e7,3,bị lỗi out khi đăng nhập samsung note s23 ultra,2026-04-16 08:45:49,6.4.1,6.4.1,vi_vn
98,bd9c1fa8-0b6c-4eca-a793-693a8fe25c9a,1,ứng dụng tồi. từ tết đến giờ k đăng nhập nổi,2026-02-24 22:37:40,6.4.1,6.4.1,vi_vn
101,e848fbee-1472-45b5-a050-7ebe95952516,1,ngày 10.02.2026 lỗi không đăng nhập được,2026-02-11 06:22:22,6.4.1,6.4.1,vi_vn
110,2e5b19b6-95a6-4096-a902-97b2962bf652,1,đăng nhập gì mà lỗi riết à,2026-01-24 23:13:13,6.4.1,6.4.1,vi_vn
150,5e10e799-e923-496b-818d-fb89540ed690,1,"đăng nhập rồi lỗi, không dùng được",2025-10-17 09:22:00,3.3.3,3.3.3,vi_vn
268,74be1267-4cc5-42d7-ad3d-6a7fc5212b86,1,"Quay như chong chóng, k đăng nhập được",2025-03-02 17:09:37,5.8.15,5.8.15,vi_vn


## Newest Reviews

In [27]:
# Date parsing is local to this cell, preventing hidden dependency on another section.
app_newest = app_store_preview.assign(_parsed_date=pd.to_datetime(app_store_df["updated_at"], utc=True, errors="coerce")).sort_values("_parsed_date", ascending=False).drop(columns="_parsed_date").head(30)
google_newest = google_play_preview.assign(_parsed_date=pd.to_datetime(google_play_df["at"], utc=True, errors="coerce")).sort_values("_parsed_date", ascending=False).drop(columns="_parsed_date").head(30)
display(app_newest)
display(google_newest)

,review_id,rating,title,content,updated_at,app_version,storefront,lang
0,14548034358,1,Liên tục lỗi không thể truy cập,"Là một người dùng lâu năm của tech, ban đầu rất hài lòng về chất lượng và giao diện của app, dễ giao dịch. Nhưng vài tháng trở lại đây app lỗi rất nhiều, chỉ đăng nhập có lúc đợi cả 5p vẫn chưa đăng nhập vào trang chủ được, thứ 2 là quảng cáo nhiều, truy cập vào được tài khoản thì phải tắt 7749 ...",2026-09-14T00:44:28-07:00,6.4.6,vn,unknown
1,14520064997,1,Tần suất bị lỗi mục tài sản quá nhiều,Rất hay bị lỗi báo k có tiền k có cổ phiếu nào trong tài khoản,2026-09-06T21:11:54-07:00,6.4.6,vn,unknown
2,14520054584,2,Khá tệ,"App rất lag, đôi khi còn không vào được",2026-09-06T21:07:12-07:00,6.4.6,vn,unknown
3,14504211053,3,Quá chậm,"Áp giao dịch gì mà chậm như rùa,mong bqt cải tiến",2026-09-02T20:32:43-07:00,6.4.6,vn,unknown
4,14468902446,1,Nghi vấn bán dữ liệu cho lái,"Tôi cầm cổ phiếu DBC 4 tháng, ngay khoảnh khắc khớp lệnh mua, cổ phiếu không một lần giá xanh, liên tiếp giảm 4 tháng liền. Tôi chấp nhận nạp tiền giữ vị thế margin, tuy nhiên sau 4 tháng, tôi muốn đổi cổ phiếu để có vị thế khác, ngay khoảnh khắc bán thì cổ phiếu DBC khớp lệnh và bay liên tiếp. ...",2026-08-24T21:00:51-07:00,6.4.4,vn,unknown
5,14467591535,5,NGUYEN VAN PHUONG,Namkuto,2026-08-24T12:29:06-07:00,6.4.4,vn,unknown
6,14465859621,1,"Giao diện rối khó sử dụng, CSKH kém, không thể gọi được NV CSKH","- UI UX kém, dễ bị bấm nhầm mua và bán\n- hotline chỉ để làm cảnh, AI trả lời tự động không giải quyết được vấn đề\n- không gọi được nv cskh, email không trả lời",2026-08-24T03:08:22-07:00,6.4.4,vn,unknown
7,14465018475,1,App lỗi quá nhiều,Nói chung quá tệ,2026-08-23T20:57:52-07:00,6.4.4,vn,unknown
8,14445501649,1,UI UX lởm quá,"UI UX này hoàn toàn ko phù hợp với iPad, chắc dev lười nên bê nguyên web sang. App chạy rất bất tiện, thao tác hoàn toàn lệch, ko hề chính xác. Dùng app thế này mà giao dịch thì lỗ nặng.",2026-08-18T22:09:21-07:00,6.4.4,vn,unknown
9,14439296592,1,Cách xác thực sự quá ngu,Lần đầu tiên tôi thấy việc xác thực để mở connection nhận data từ websocket nó ngớ ngẩn như vậy. Cứ 24 tiếng lại phải chạy vô app lấy cái OTP bằng tay xong đi cập nhật biến môi trường trước giờ giao dịch để k bị miss data. Đều đặn hằng ngày. Dev bên này nên thử mở cái docs của DNSE với SSi họ là...,2026-08-17T08:52:20-07:00,6.4.4,vn,unknown


,reviewId,score,content,at,reviewCreatedVersion,appVersion,collection_locale
0,6638c299-2864-4386-837e-41064b75be48,1,sao cứ cứng đơ không nhích nhút gõ bất kỳ một chữ nào được....,2026-09-21 12:13:27,NaN,NaN,vi_vn
1,45b7bb0f-cd74-43a9-9b85-bd050b6a1f9f,1,Mới dùng thử . App giao diên quá chán. Dùng tcbs . Mà phải vào ssi xem giá . .ap quá chán luôn .,2026-09-21 05:51:18,6.4.5,6.4.5,vi_vn
2,4ae66065-e785-4516-a20d-51040c7a80d2,1,"rẻ rách, đã rườm rà còn khó xài",2026-09-18 18:49:22,6.4.5,6.4.5,vi_vn
3,f93f9a8e-c094-4ceb-8268-f4ba73dbb18c,1,"bắt làm chữ ký số mà cứ chụp ảnh xong là đơ, ko làm đc cái gì nữa, xóa đi cài lại vẫn thế",2026-09-15 22:04:51,6.4.5,6.4.5,vi_vn
4,ef9573f5-cac7-4b27-857a-f29bebf2c4d2,3,"Tôi đang sở hữu quỹ tcef đặt mua thêm ccq ngay trong tcef mà khi đặt lệnh lại lòi ra lệnh mua tcbf là sao vậy, ai code app kỳ vậy",2026-09-14 08:56:29,6.4.5,6.4.5,vi_vn
5,b60631dc-52f7-4a6e-ab62-777981d617ec,1,"Giao diện app quá tệ. Tính năng cơ bản như xem chart và đặt lệnh chuyển qua lại rất phức tạp, tôi muốn vừa xem chart nhanh vừa đặt lệnh nhiều ngày (24/7) đơn giản, mà app quá rườm rà, giao diện nhiều tính năng rác không dùng đến chiếm hết chỗ. Muốn xem danh mục của tôi đang giữ biến động thế nào...",2026-09-11 15:57:15,6.4.5,6.4.5,vi_vn
6,7d25eb7f-ac40-4de4-8943-5ab02a03cdbf,1,"App Techcombank Mobile nhanh bao nhiêu thì app TCBS chậm bấy nhiêu, đó là vì TCB mobile có giao diện đơn giản. Để cải thiện, các bạn cần đơn giản hóa giao diện, các cửa sổ lựa chọn chỉ nên hiện những gì cần thiết, cơ bản (các tính năng thông minh khác dành cho nhà đầu tư chuyên nghiệp nên được c...",2026-09-09 06:36:39,NaN,NaN,vi_vn
7,84a4e782-1615-4764-ad24-3974c5da8eb0,2,app lag giật khó dùng,2026-09-08 08:36:53,6.4.5,6.4.5,vi_vn
2712,7fd64ff8-fbd4-4520-a656-c439f9e4443d,1,"Đội ngũ hỗ trợ xử lý chậm chạp, lề mề. Tài khoản khách hàng cần mở khóa, phải chạy lên chi nhánh TCB kí giấy. Rùi phải đợi 2-3 ngày vận chuyển giấy tờ từ TCB ra TCBS chi nhánh HN. Sau đó phải đợi xử lý tờ giấy, rùi đợi 24 giờ để mở khóa. Loằng ngoằng như thời cổ đại.",2026-09-05 11:12:28,NaN,NaN,en_vn
8,74c2b3de-07a9-4e83-afe4-76ce8134f8f0,1,"Rất tệ yêu cầu mã iOTP mà chờ gần 30p không có , liên hệ nhân viên thì không được , hệ thống quản lý quá kém",2026-08-29 21:30:32,6.4.5,6.4.5,vi_vn


## Oldest Reviews

In [28]:
# Date parsing is local to this cell, preventing hidden dependency on another section.
app_oldest = app_store_preview.assign(_parsed_date=pd.to_datetime(app_store_df["updated_at"], utc=True, errors="coerce")).sort_values("_parsed_date").drop(columns="_parsed_date").head(30)
google_oldest = google_play_preview.assign(_parsed_date=pd.to_datetime(google_play_df["at"], utc=True, errors="coerce")).sort_values("_parsed_date").drop(columns="_parsed_date").head(30)
display(app_oldest)
display(google_oldest)

,review_id,rating,title,content,updated_at,app_version,storefront,lang
521,9819441432,3,Hay bị lỗi app,Cứ vài ngày là lỗi app phãi xoá đi tải lại hoài.,2023-04-13T20:22:33-07:00,5.0.1,vn,unknown
520,10191157286,4,Máy bị xoay ngang ko xoay lại đc,"macbook pro m1 app bị lỗi, bị xoay ngang ko xoay lại đc",2023-07-28T03:58:35-07:00,5.0.1,vn,unknown
519,10217332140,3,Nâng cấp phải giữ bản cũ,"Nâng cấp phải giữ bản cũ, tôi dùng ios 12.5.7 không vào được! Giờ phải thay điện thoại! Các app bên Cty Ck khác đều dùng được",2023-08-03T20:54:21-07:00,5.1,vn,unknown
518,10383443236,5,TCBS,"App rất tuyệt vời, mọi giao dịch đều nhanh chóng chính xác, đánh giá trên cả 5 sao!\nTài khoản TCBS: 105CA07444",2023-09-18T22:03:19-07:00,5.1,vn,unknown
517,11124085882,3,không đổi được mật khẩu,"tạo tài khoản xong, vào ứng dụng bắt đổi mật khẩu mãi mà hok đổi được. admin kiểm tra lại giùm mình với",2024-04-05T05:18:45-07:00,5.5,vn,unknown
516,11708342893,1,Tệ,Dịch vụ cskh trực tuyến tệ! Xử lý vấn đề trên app tệ! Lỗi app tệ,2024-09-10T00:01:09-07:00,5.7,vn,unknown
515,11736875655,5,Hữu ích,Giao dịch tiện lợi và nhanh chóng,2024-09-18T03:02:07-07:00,5.7,vn,unknown
514,11736877477,5,Rất là ok,Nhanh gọn thuận tiện,2024-09-18T03:02:55-07:00,5.7,vn,unknown
513,11736879163,5,App rất dễ sử dụng,Nhiều tiện ích Stk 105CH91198,2024-09-18T03:03:40-07:00,5.7,vn,unknown
512,11736881714,5,105CP01787,105CP01787 tôi thích bảng giá,2024-09-18T03:04:46-07:00,5.7,vn,unknown


,reviewId,score,content,at,reviewCreatedVersion,appVersion,collection_locale
3222,d9cf435b-e857-4894-9aaf-363abf995b71,5,Very convenience to trade on Vietnam stock market,2015-08-13 11:10:30,1.0,1.0,en_vn
3221,971758e7-1b59-445f-941f-e44c69810165,5,Clean and Fast. Menu needs to improve for more convenient.,2015-08-24 15:06:49,NaN,NaN,en_vn
2711,0f1413df-783e-42a6-8532-e1654d507521,5,Ok,2015-09-02 11:07:10,1.0,1.0,vi_vn
3220,e8718449-090f-4cbd-8471-9b767f2065fc,1,Phần mềm không cài đc trên note 4 :(,2015-09-13 19:32:39,NaN,NaN,en_vn
2710,d916d16d-74cd-41ca-86ca-fb57447b18da,4,Phần mềm cập nhật thông tin chậm. Đăng nhập đôi lúc k vào được. Tài khoản tổng tính hay bị sai khi giá cổ phiếu tăng lên hay giảm xuống. Phần trăm lãi bị nhầm lẫn khi chuyển cổ phiếu qua lại giữa thường và ký quỹ. Nhìn chung ứng dụng này chưa tốt cả về sự mượt mà lẫn cập nhật thông tin thị trường.,2016-08-14 08:24:52,1.1,1.1,vi_vn
2709,18ab7cb9-5b97-4365-a57e-882af1ce245a,4,"Thông tin cập nhật hay bị sai, giá cp lên xuống ko được cập nhật đúng. Không cho bán lô lẻ như FPTS. Hy vọng TCBs sớm fix lỗi.",2016-09-19 11:33:26,1.1,1.1,vi_vn
3219,0de9b206-e6bf-4af4-9c22-b5c7384f77b3,1,Vừa đăng nhập xong 2' thì tự quit,2016-10-25 14:46:57,1.1.1,1.1.1,en_vn
2708,9bbd9691-fab8-4b68-9aad-24f1f40c0742,2,Cập nhật chậm. Tính sai tổng tài sản. Đề nghị cập nhật,2017-07-12 13:02:47,1.1.2,1.1.2,vi_vn
2707,f98616ef-454c-4196-a4fb-65d9ffc1ae97,2,"1. Bảng giá hiển thị bị làm tròn số. Vd: giá 39.15 làm tròn thành 39.2 2. Danh mục đầu tư chưa tính lãi/lỗ tổng cộng 3. Không có thông tin về cổ phiếu ( biểu đồ giá + klg, báo cáo tài chính: tổng quan + chỉ tiêu + chỉ số, tin tức thị trường liên quan,...) 4....",2017-10-09 11:09:47,1.1.6,1.1.6,vi_vn
3218,27976d92-a850-4c7f-bc10-aa36403ade07,1,"Cài đặt xong nhưng không thể đăng nhập sử dụng, liên tục báo lỗi connection failed, mặc dù đăng nhập bằng account & code đang sử dụng trên bản web.",2017-12-02 10:39:36,1.1.6,1.1.6,en_vn


## App Store Inspection

In [29]:
APP_STORE_INSPECTION_RATING = 1  # Change to 1, 2, 3, 4, or 5.
app_inspection = app_store_preview.assign(_parsed_date=pd.to_datetime(app_store_df["updated_at"], utc=True, errors="coerce"))
app_checks = pd.Series({
    "rows": len(app_store_df),
    "unique_review_id": app_store_df["review_id"].nunique(),
    "duplicate_review_id_records": int(app_store_df["review_id"].duplicated(keep=False).sum()),
    "missing_content": int(app_store_df["content"].isna().sum()),
    "missing_rating": int(app_store_df["rating"].isna().sum()),
    "missing_updated_at": int(app_store_df["updated_at"].isna().sum()),
})
display(app_checks.rename("count").to_frame())
print("Random 20")
display(app_store_preview.sample(n=20, random_state=42))
print("Earliest 20")
display(app_inspection.sort_values("_parsed_date").drop(columns="_parsed_date").head(20))
print("Latest 20")
display(app_inspection.sort_values("_parsed_date", ascending=False).drop(columns="_parsed_date").head(20))
print(f"Rating {APP_STORE_INSPECTION_RATING}: up to 20")
display(app_store_preview[pd.to_numeric(app_store_preview["rating"], errors="coerce").eq(APP_STORE_INSPECTION_RATING)].head(20))

,count
rows,522
unique_review_id,522
duplicate_review_id_records,0
missing_content,0
missing_rating,0
missing_updated_at,0


Random 20


,review_id,rating,title,content,updated_at,app_version,storefront,lang
434,11737328061,5,Tuyệt vời,105CA18669,2024-09-18T05:47:47-07:00,5.7,vn,unknown
440,11737258935,5,Dễ sử dụng,"Giao diện app dễ sử dụng, đẹp",2024-09-18T05:26:06-07:00,5.7,vn,unknown
6,14465859621,1,"Giao diện rối khó sử dụng, CSKH kém, không thể gọi được NV CSKH","- UI UX kém, dễ bị bấm nhầm mua và bán\n- hotline chỉ để làm cảnh, AI trả lời tự động không giải quyết được vấn đề\n- không gọi được nv cskh, email không trả lời",2026-08-24T03:08:22-07:00,6.4.4,vn,unknown
184,13014069107,1,lỗi chuyển tiền,k rút tiền dc,2025-08-14T01:51:40-07:00,6.0.2,vn,unknown
78,13835758727,1,TK ng thân không nộp được tiền,"Mình tạo tk con, Lần 1 nộp bằng cú pháp khi nhấn nộp tiền trên áp, 2 ngày sau trả lại báo sai cú pháp, trong khi cú pháp tự đông gen ra\n\nLần này 4 ngày rồi không back lại tiền, mà back lại cũng k biết làm thế nào để nộp",2026-03-10T21:52:51-07:00,6.4.2,vn,unknown
299,12092046546,2,Phần đặt lệnh cổ phiếu thực sự chán. Đơn điệu không hiển được thông tin thị trường thời điểm đó.,"Nhiều lúc đặt giá xong mở ra xem giá tt đã thay đổi từ lúc nào. Phần đặt khối lượng cũng vậy, nhiều lúc quên ko biết khối lượng cp của mình có bao nhiêu lại phải quay lại từ đầu để xem. Thực sự rất khó chịu",2024-12-22T01:52:10-07:00,5.8.2,vn,unknown
521,9819441432,3,Hay bị lỗi app,Cứ vài ngày là lỗi app phãi xoá đi tải lại hoài.,2023-04-13T20:22:33-07:00,5.0.1,vn,unknown
484,11736926631,5,Dich vu tot,105CH90456,2024-09-18T03:24:09-07:00,5.7,vn,unknown
117,13630948771,1,Lỗi ai chịu,"Đang lỗi không giao dịch được, hãy sử lý và có lời giải thích thoả đáng. Lỗi hơn 1 tiếng chưa có trả lời khách hàng",2026-01-13T19:07:52-07:00,6.3,vn,unknown
137,13500073382,2,Biểu đồ kỹ thuật,Biểu đồ kỹ thuật chỉ cho xem theo hình dọc hoặc quay ngang thì không được full màn xem rất khó chịu. TCBS khắc phục thêm quả này nữa thì ngon,2025-12-10T22:23:18-07:00,6.2,vn,unknown


Earliest 20


,review_id,rating,title,content,updated_at,app_version,storefront,lang
521,9819441432,3,Hay bị lỗi app,Cứ vài ngày là lỗi app phãi xoá đi tải lại hoài.,2023-04-13T20:22:33-07:00,5.0.1,vn,unknown
520,10191157286,4,Máy bị xoay ngang ko xoay lại đc,"macbook pro m1 app bị lỗi, bị xoay ngang ko xoay lại đc",2023-07-28T03:58:35-07:00,5.0.1,vn,unknown
519,10217332140,3,Nâng cấp phải giữ bản cũ,"Nâng cấp phải giữ bản cũ, tôi dùng ios 12.5.7 không vào được! Giờ phải thay điện thoại! Các app bên Cty Ck khác đều dùng được",2023-08-03T20:54:21-07:00,5.1,vn,unknown
518,10383443236,5,TCBS,"App rất tuyệt vời, mọi giao dịch đều nhanh chóng chính xác, đánh giá trên cả 5 sao!\nTài khoản TCBS: 105CA07444",2023-09-18T22:03:19-07:00,5.1,vn,unknown
517,11124085882,3,không đổi được mật khẩu,"tạo tài khoản xong, vào ứng dụng bắt đổi mật khẩu mãi mà hok đổi được. admin kiểm tra lại giùm mình với",2024-04-05T05:18:45-07:00,5.5,vn,unknown
516,11708342893,1,Tệ,Dịch vụ cskh trực tuyến tệ! Xử lý vấn đề trên app tệ! Lỗi app tệ,2024-09-10T00:01:09-07:00,5.7,vn,unknown
515,11736875655,5,Hữu ích,Giao dịch tiện lợi và nhanh chóng,2024-09-18T03:02:07-07:00,5.7,vn,unknown
514,11736877477,5,Rất là ok,Nhanh gọn thuận tiện,2024-09-18T03:02:55-07:00,5.7,vn,unknown
513,11736879163,5,App rất dễ sử dụng,Nhiều tiện ích Stk 105CH91198,2024-09-18T03:03:40-07:00,5.7,vn,unknown
512,11736881714,5,105CP01787,105CP01787 tôi thích bảng giá,2024-09-18T03:04:46-07:00,5.7,vn,unknown


Latest 20


,review_id,rating,title,content,updated_at,app_version,storefront,lang
0,14548034358,1,Liên tục lỗi không thể truy cập,"Là một người dùng lâu năm của tech, ban đầu rất hài lòng về chất lượng và giao diện của app, dễ giao dịch. Nhưng vài tháng trở lại đây app lỗi rất nhiều, chỉ đăng nhập có lúc đợi cả 5p vẫn chưa đăng nhập vào trang chủ được, thứ 2 là quảng cáo nhiều, truy cập vào được tài khoản thì phải tắt 7749 ...",2026-09-14T00:44:28-07:00,6.4.6,vn,unknown
1,14520064997,1,Tần suất bị lỗi mục tài sản quá nhiều,Rất hay bị lỗi báo k có tiền k có cổ phiếu nào trong tài khoản,2026-09-06T21:11:54-07:00,6.4.6,vn,unknown
2,14520054584,2,Khá tệ,"App rất lag, đôi khi còn không vào được",2026-09-06T21:07:12-07:00,6.4.6,vn,unknown
3,14504211053,3,Quá chậm,"Áp giao dịch gì mà chậm như rùa,mong bqt cải tiến",2026-09-02T20:32:43-07:00,6.4.6,vn,unknown
4,14468902446,1,Nghi vấn bán dữ liệu cho lái,"Tôi cầm cổ phiếu DBC 4 tháng, ngay khoảnh khắc khớp lệnh mua, cổ phiếu không một lần giá xanh, liên tiếp giảm 4 tháng liền. Tôi chấp nhận nạp tiền giữ vị thế margin, tuy nhiên sau 4 tháng, tôi muốn đổi cổ phiếu để có vị thế khác, ngay khoảnh khắc bán thì cổ phiếu DBC khớp lệnh và bay liên tiếp. ...",2026-08-24T21:00:51-07:00,6.4.4,vn,unknown
5,14467591535,5,NGUYEN VAN PHUONG,Namkuto,2026-08-24T12:29:06-07:00,6.4.4,vn,unknown
6,14465859621,1,"Giao diện rối khó sử dụng, CSKH kém, không thể gọi được NV CSKH","- UI UX kém, dễ bị bấm nhầm mua và bán\n- hotline chỉ để làm cảnh, AI trả lời tự động không giải quyết được vấn đề\n- không gọi được nv cskh, email không trả lời",2026-08-24T03:08:22-07:00,6.4.4,vn,unknown
7,14465018475,1,App lỗi quá nhiều,Nói chung quá tệ,2026-08-23T20:57:52-07:00,6.4.4,vn,unknown
8,14445501649,1,UI UX lởm quá,"UI UX này hoàn toàn ko phù hợp với iPad, chắc dev lười nên bê nguyên web sang. App chạy rất bất tiện, thao tác hoàn toàn lệch, ko hề chính xác. Dùng app thế này mà giao dịch thì lỗ nặng.",2026-08-18T22:09:21-07:00,6.4.4,vn,unknown
9,14439296592,1,Cách xác thực sự quá ngu,Lần đầu tiên tôi thấy việc xác thực để mở connection nhận data từ websocket nó ngớ ngẩn như vậy. Cứ 24 tiếng lại phải chạy vô app lấy cái OTP bằng tay xong đi cập nhật biến môi trường trước giờ giao dịch để k bị miss data. Đều đặn hằng ngày. Dev bên này nên thử mở cái docs của DNSE với SSi họ là...,2026-08-17T08:52:20-07:00,6.4.4,vn,unknown


Rating 1: up to 20


,review_id,rating,title,content,updated_at,app_version,storefront,lang
0,14548034358,1,Liên tục lỗi không thể truy cập,"Là một người dùng lâu năm của tech, ban đầu rất hài lòng về chất lượng và giao diện của app, dễ giao dịch. Nhưng vài tháng trở lại đây app lỗi rất nhiều, chỉ đăng nhập có lúc đợi cả 5p vẫn chưa đăng nhập vào trang chủ được, thứ 2 là quảng cáo nhiều, truy cập vào được tài khoản thì phải tắt 7749 ...",2026-09-14T00:44:28-07:00,6.4.6,vn,unknown
1,14520064997,1,Tần suất bị lỗi mục tài sản quá nhiều,Rất hay bị lỗi báo k có tiền k có cổ phiếu nào trong tài khoản,2026-09-06T21:11:54-07:00,6.4.6,vn,unknown
4,14468902446,1,Nghi vấn bán dữ liệu cho lái,"Tôi cầm cổ phiếu DBC 4 tháng, ngay khoảnh khắc khớp lệnh mua, cổ phiếu không một lần giá xanh, liên tiếp giảm 4 tháng liền. Tôi chấp nhận nạp tiền giữ vị thế margin, tuy nhiên sau 4 tháng, tôi muốn đổi cổ phiếu để có vị thế khác, ngay khoảnh khắc bán thì cổ phiếu DBC khớp lệnh và bay liên tiếp. ...",2026-08-24T21:00:51-07:00,6.4.4,vn,unknown
6,14465859621,1,"Giao diện rối khó sử dụng, CSKH kém, không thể gọi được NV CSKH","- UI UX kém, dễ bị bấm nhầm mua và bán\n- hotline chỉ để làm cảnh, AI trả lời tự động không giải quyết được vấn đề\n- không gọi được nv cskh, email không trả lời",2026-08-24T03:08:22-07:00,6.4.4,vn,unknown
7,14465018475,1,App lỗi quá nhiều,Nói chung quá tệ,2026-08-23T20:57:52-07:00,6.4.4,vn,unknown
8,14445501649,1,UI UX lởm quá,"UI UX này hoàn toàn ko phù hợp với iPad, chắc dev lười nên bê nguyên web sang. App chạy rất bất tiện, thao tác hoàn toàn lệch, ko hề chính xác. Dùng app thế này mà giao dịch thì lỗ nặng.",2026-08-18T22:09:21-07:00,6.4.4,vn,unknown
9,14439296592,1,Cách xác thực sự quá ngu,Lần đầu tiên tôi thấy việc xác thực để mở connection nhận data từ websocket nó ngớ ngẩn như vậy. Cứ 24 tiếng lại phải chạy vô app lấy cái OTP bằng tay xong đi cập nhật biến môi trường trước giờ giao dịch để k bị miss data. Đều đặn hằng ngày. Dev bên này nên thử mở cái docs của DNSE với SSi họ là...,2026-08-17T08:52:20-07:00,6.4.4,vn,unknown
10,14425846977,1,"Làm ăn ko kỹ càng, cẩu thả","Các công ty đã rời sàn như HLG thì bị tcinvest loại luôn khỏi list, làm nhà đầu tư mất luôn thông tin số lượng cổ phiếu đang sở hữu. Ko thể nào biết để gặp cty HLG làm giấy xác nhận. Nhiều mã rời sàn như hiện nay thì mất rất nhiều. List tôi gồm 100 loại cổ phiếu nên ko nhớ được",2026-08-14T01:14:42-07:00,6.4.4,vn,unknown
12,14418152911,1,Quá phiền hà,"Phục vụ quá cứng nhắc thay số điện thoại, cccd bắt yêu cầu mở thẻ ngân hàng để xác minh thông tin.",2026-08-12T03:46:04-07:00,6.4.4,vn,unknown
13,14414751396,1,App như cặc,App như lol mỗi vào để điền thông tin xác thực thôi mà đen xì ko hiện ra 1 cái gì . Ngân hàng lớn mà thuê team IT trình độ kém quá,2026-08-11T07:10:45-07:00,6.4.4,vn,unknown


## Google Play Inspection

In [30]:
google_inspection = google_play_preview.assign(_parsed_date=pd.to_datetime(google_play_df["at"], utc=True, errors="coerce"))
google_checks = pd.Series({
    "rows": len(google_play_df),
    "unique_reviewId": google_play_df["reviewId"].nunique(),
    "duplicate_reviewId_records": int(google_play_df["reviewId"].duplicated(keep=False).sum()),
    "missing_content": int(google_play_df["content"].isna().sum()),
    "missing_score": int(google_play_df["score"].isna().sum()),
    "missing_at": int(google_play_df["at"].isna().sum()),
    "missing_reviewCreatedVersion": int(google_play_df["reviewCreatedVersion"].isna().sum()),
    "missing_appVersion": int(google_play_df["appVersion"].isna().sum()),
})
display(google_checks.rename("count").to_frame())
assert google_checks["missing_appVersion"] == 459
print("Collection locale counts")
display(google_play_df["collection_locale"].value_counts(dropna=False).rename("count").to_frame())
print("Random 20")
display(google_play_preview.sample(n=20, random_state=42))
print("Earliest 20")
display(google_inspection.sort_values("_parsed_date").drop(columns="_parsed_date").head(20))
print("Latest 20")
display(google_inspection.sort_values("_parsed_date", ascending=False).drop(columns="_parsed_date").head(20))

,count
rows,3223
unique_reviewId,3223
duplicate_reviewId_records,0
missing_content,0
missing_score,0
missing_at,0
missing_reviewCreatedVersion,459
missing_appVersion,459


Collection locale counts


,count
collection_locale,
vi_vn,2712
en_vn,511


Random 20


,reviewId,score,content,at,reviewCreatedVersion,appVersion,collection_locale
299,abff860a-3329-408e-a188-cf64dac8a665,1,"Ứng dụng rối nuồi khó sử dụng, lag, lỗi",2025-01-01 16:09:11,5.8.11,5.8.11,vi_vn
501,5db4f5b8-b014-46c1-ade1-9220e003ca81,1,Rối rắm. Khó dùng. Lag. Chậm. Nếu có 0 sao thì 1 sao cũng k xứng đáng,2024-08-02 09:27:56,NaN,NaN,vi_vn
631,08de7a4a-fecb-4600-a727-064057aee775,5,105C0910TA tôi thích bảng giá,2024-03-04 21:45:44,5.4.1,5.4.1,vi_vn
2098,9229dec5-3fce-4985-b83a-8b5da1fb7817,2,Lúc nào cũng gặp trục trặc với iOTP,2021-08-25 23:26:43,3.1.1,3.1.1,vi_vn
1377,b840a5c7-9bbb-4e6f-badd-e6c1fd2f5d4a,1,Cài hoài ko được,2022-04-26 19:56:17,NaN,NaN,vi_vn
1709,cc027484-9281-42c5-a44f-5bb297451ded,1,Ức chế những lúc thị trường điều chỉnh mạnh. Thích là không cho đặt lệnh. Buồn buồn là chơi lớn cấm cửa không cho giao dịch. Quá tệ.,2021-11-23 11:41:12,3.1.1,3.1.1,vi_vn
564,b9dac638-cc17-4d03-a456-ccf173c9464d,3,"Xoay ngang chưa được tốt lắm, dọc thì full màn, cần bổ sung thêm chỉ báo vào biểu đồ để có thể thuận tiện phân tích. Đa phần ở mức cơ bản",2024-04-03 00:17:36,5.5.2,5.5.2,vi_vn
1323,ed66daf9-cc37-4ec8-a2f0-a95c6580beb1,1,"App cực ngu, tìm kiếm, sao kê tài khoản, thực hiện margin tìm tất cả danh mục không thấy , sắp xếp rối tít mù . Mặc dù cũng dân công nghệ mà ko dùng nổi cái app này",2022-07-18 19:26:33,3.3.2,3.3.2,vi_vn
3060,d1e182cd-53a7-4a66-b8b1-de6afda48329,2,Please update app tobe able modify/cancel sell/buy command base on stock exchange. It should not lock all command on other stock like HNX UpCom,2021-06-10 15:01:54,3.0,3.0,en_vn
438,166a3e51-08f2-444f-adc9-edf731626ca0,5,Đâu tư tài chính,2024-08-16 07:59:48,5.7,5.7,vi_vn


Earliest 20


,reviewId,score,content,at,reviewCreatedVersion,appVersion,collection_locale
3222,d9cf435b-e857-4894-9aaf-363abf995b71,5,Very convenience to trade on Vietnam stock market,2015-08-13 11:10:30,1.0,1.0,en_vn
3221,971758e7-1b59-445f-941f-e44c69810165,5,Clean and Fast. Menu needs to improve for more convenient.,2015-08-24 15:06:49,NaN,NaN,en_vn
2711,0f1413df-783e-42a6-8532-e1654d507521,5,Ok,2015-09-02 11:07:10,1.0,1.0,vi_vn
3220,e8718449-090f-4cbd-8471-9b767f2065fc,1,Phần mềm không cài đc trên note 4 :(,2015-09-13 19:32:39,NaN,NaN,en_vn
2710,d916d16d-74cd-41ca-86ca-fb57447b18da,4,Phần mềm cập nhật thông tin chậm. Đăng nhập đôi lúc k vào được. Tài khoản tổng tính hay bị sai khi giá cổ phiếu tăng lên hay giảm xuống. Phần trăm lãi bị nhầm lẫn khi chuyển cổ phiếu qua lại giữa thường và ký quỹ. Nhìn chung ứng dụng này chưa tốt cả về sự mượt mà lẫn cập nhật thông tin thị trường.,2016-08-14 08:24:52,1.1,1.1,vi_vn
2709,18ab7cb9-5b97-4365-a57e-882af1ce245a,4,"Thông tin cập nhật hay bị sai, giá cp lên xuống ko được cập nhật đúng. Không cho bán lô lẻ như FPTS. Hy vọng TCBs sớm fix lỗi.",2016-09-19 11:33:26,1.1,1.1,vi_vn
3219,0de9b206-e6bf-4af4-9c22-b5c7384f77b3,1,Vừa đăng nhập xong 2' thì tự quit,2016-10-25 14:46:57,1.1.1,1.1.1,en_vn
2708,9bbd9691-fab8-4b68-9aad-24f1f40c0742,2,Cập nhật chậm. Tính sai tổng tài sản. Đề nghị cập nhật,2017-07-12 13:02:47,1.1.2,1.1.2,vi_vn
2707,f98616ef-454c-4196-a4fb-65d9ffc1ae97,2,"1. Bảng giá hiển thị bị làm tròn số. Vd: giá 39.15 làm tròn thành 39.2 2. Danh mục đầu tư chưa tính lãi/lỗ tổng cộng 3. Không có thông tin về cổ phiếu ( biểu đồ giá + klg, báo cáo tài chính: tổng quan + chỉ tiêu + chỉ số, tin tức thị trường liên quan,...) 4....",2017-10-09 11:09:47,1.1.6,1.1.6,vi_vn
3218,27976d92-a850-4c7f-bc10-aa36403ade07,1,"Cài đặt xong nhưng không thể đăng nhập sử dụng, liên tục báo lỗi connection failed, mặc dù đăng nhập bằng account & code đang sử dụng trên bản web.",2017-12-02 10:39:36,1.1.6,1.1.6,en_vn


Latest 20


,reviewId,score,content,at,reviewCreatedVersion,appVersion,collection_locale
0,6638c299-2864-4386-837e-41064b75be48,1,sao cứ cứng đơ không nhích nhút gõ bất kỳ một chữ nào được....,2026-09-21 12:13:27,NaN,NaN,vi_vn
1,45b7bb0f-cd74-43a9-9b85-bd050b6a1f9f,1,Mới dùng thử . App giao diên quá chán. Dùng tcbs . Mà phải vào ssi xem giá . .ap quá chán luôn .,2026-09-21 05:51:18,6.4.5,6.4.5,vi_vn
2,4ae66065-e785-4516-a20d-51040c7a80d2,1,"rẻ rách, đã rườm rà còn khó xài",2026-09-18 18:49:22,6.4.5,6.4.5,vi_vn
3,f93f9a8e-c094-4ceb-8268-f4ba73dbb18c,1,"bắt làm chữ ký số mà cứ chụp ảnh xong là đơ, ko làm đc cái gì nữa, xóa đi cài lại vẫn thế",2026-09-15 22:04:51,6.4.5,6.4.5,vi_vn
4,ef9573f5-cac7-4b27-857a-f29bebf2c4d2,3,"Tôi đang sở hữu quỹ tcef đặt mua thêm ccq ngay trong tcef mà khi đặt lệnh lại lòi ra lệnh mua tcbf là sao vậy, ai code app kỳ vậy",2026-09-14 08:56:29,6.4.5,6.4.5,vi_vn
5,b60631dc-52f7-4a6e-ab62-777981d617ec,1,"Giao diện app quá tệ. Tính năng cơ bản như xem chart và đặt lệnh chuyển qua lại rất phức tạp, tôi muốn vừa xem chart nhanh vừa đặt lệnh nhiều ngày (24/7) đơn giản, mà app quá rườm rà, giao diện nhiều tính năng rác không dùng đến chiếm hết chỗ. Muốn xem danh mục của tôi đang giữ biến động thế nào...",2026-09-11 15:57:15,6.4.5,6.4.5,vi_vn
6,7d25eb7f-ac40-4de4-8943-5ab02a03cdbf,1,"App Techcombank Mobile nhanh bao nhiêu thì app TCBS chậm bấy nhiêu, đó là vì TCB mobile có giao diện đơn giản. Để cải thiện, các bạn cần đơn giản hóa giao diện, các cửa sổ lựa chọn chỉ nên hiện những gì cần thiết, cơ bản (các tính năng thông minh khác dành cho nhà đầu tư chuyên nghiệp nên được c...",2026-09-09 06:36:39,NaN,NaN,vi_vn
7,84a4e782-1615-4764-ad24-3974c5da8eb0,2,app lag giật khó dùng,2026-09-08 08:36:53,6.4.5,6.4.5,vi_vn
2712,7fd64ff8-fbd4-4520-a656-c439f9e4443d,1,"Đội ngũ hỗ trợ xử lý chậm chạp, lề mề. Tài khoản khách hàng cần mở khóa, phải chạy lên chi nhánh TCB kí giấy. Rùi phải đợi 2-3 ngày vận chuyển giấy tờ từ TCB ra TCBS chi nhánh HN. Sau đó phải đợi xử lý tờ giấy, rùi đợi 24 giờ để mở khóa. Loằng ngoằng như thời cổ đại.",2026-09-05 11:12:28,NaN,NaN,en_vn
8,74c2b3de-07a9-4e83-afe4-76ce8134f8f0,1,"Rất tệ yêu cầu mã iOTP mà chờ gần 30p không có , liên hệ nhân viên thì không được , hệ thống quản lý quá kém",2026-08-29 21:30:32,6.4.5,6.4.5,vi_vn


## Very Short Reviews

For consistency with the cleaning audit, character length is calculated on a temporary display-only string after trimming and collapsing whitespace. The rule is `length <= 10`. No raw value is changed or removed. A short review such as "Tệ", "Good", "Ok", or "Tốt" can still be valid.

In [31]:
def qa_text_length(series):
    return series.astype("string").str.replace(r"\s+", " ", regex=True).str.strip().str.len()

app_short_mask = qa_text_length(app_store_df["content"]).le(10)
google_short_mask = qa_text_length(google_play_df["content"]).le(10)
assert int(app_short_mask.sum()) == 57
assert int(google_short_mask.sum()) == 476
print(f"App Store very short reviews: {int(app_short_mask.sum()):,}")
display(app_store_preview[app_short_mask].head(50))
print(f"Google Play very short reviews: {int(google_short_mask.sum()):,}")
display(google_play_preview[google_short_mask].head(50))

App Store very short reviews: 57


,review_id,rating,title,content,updated_at,app_version,storefront,lang
5,14467591535,5,NGUYEN VAN PHUONG,Namkuto,2026-08-24T12:29:06-07:00,6.4.4,vn,unknown
27,14142865973,1,Ai như tôi không nạo tiền vào không lâu tiền bay hết sạch,Chắn,2026-06-04T06:07:36-07:00,6.4.4,vn,unknown
34,14009628721,1,Lỗi hiển thị sai giá ko thể mua bán,App lỗi tệ,2026-04-29T08:34:18-07:00,6.4.3,vn,unknown
47,14007820033,1,app lởm,lởm vl,2026-04-28T19:37:47-07:00,6.4.3,vn,unknown
95,13732893373,1,Nạo tiền không và tài khoản,Bad,2026-02-09T21:05:11-07:00,6.4.1,vn,unknown
160,13276981462,1,App thường xuyên treo lúc cần bán,Tệ,2025-10-17T00:44:07-07:00,6.1,vn,unknown
192,12964759767,1,App gì mà lag lộn lên,1 điểm,2025-08-01T15:41:55-07:00,6.0,vn,unknown
201,12889467768,2,Lỗi,Hay bị lỗi,2025-07-13T23:07:33-07:00,6.0,vn,unknown
230,12729702813,5,Good app,Good app,2025-06-02T20:41:08-07:00,5.9,vn,unknown
231,12727113643,5,105CTOIYEUVN tôi thích bảng giá,Rất hay,2025-06-02T05:18:56-07:00,5.9,vn,unknown


Google Play very short reviews: 476


,reviewId,score,content,at,reviewCreatedVersion,appVersion,collection_locale
15,01ad6ded-2aab-45a5-b0c7-e94d5828b921,1,lag,2026-08-07 09:24:22,6.4.1,6.4.1,vi_vn
20,07616126-4fd6-4860-a991-a0a6ea51f192,5,mong đợi,2026-07-24 10:33:14,6.4.5,6.4.5,vi_vn
21,56f40fd1-9968-4020-a53d-7faa5f81b4ba,5,good,2026-07-16 08:30:18,5.8.13,5.8.13,vi_vn
49,028fcd26-9ed7-4edb-a256-5d54363dfa43,1,DV CSKH tệ,2026-05-15 13:12:47,6.4.3,6.4.3,vi_vn
64,8832565b-75c1-45d2-8fc2-a846e4f983be,5,dễ dùng đó,2026-04-17 11:21:53,6.4.1,6.4.1,vi_vn
76,12a15533-34a5-4b70-8535-3bee8c3a3f30,5,ok,2026-04-15 15:39:08,6.4.1,6.4.1,vi_vn
95,eddf39d9-bd84-4593-8b55-0d8756ca835e,1,ap lỏ vl,2026-03-01 19:01:57,6.4.1,6.4.1,vi_vn
99,729f405c-971f-47b7-803d-ed285f1e2299,5,ok,2026-02-22 01:15:19,6.4.1,6.4.1,vi_vn
166,e17a1244-02e2-46a7-b4e8-ef5e21919c2c,1,ko vào dc,2025-09-15 10:29:27,6.2,6.2,vi_vn
174,a604cb52-b613-43b7-98e2-58a855dd280c,5,rất ok,2025-09-05 16:08:32,6.2,6.2,vi_vn


## Exact Duplicate Text Inspection

Exact duplicate text is flagged across the current combined raw inputs. Distinct review IDs are retained: identical wording does not prove that reviews or users are duplicates.

In [32]:
text_index = pd.concat([
    app_store_df[["review_id", "content"]].rename(columns={"content": "review_text_raw"}).assign(source="App Store"),
    google_play_df[["reviewId", "content"]].rename(columns={"reviewId": "review_id", "content": "review_text_raw"}).assign(source="Google Play"),
], ignore_index=True)
exact_text_counts = text_index["review_text_raw"].value_counts(dropna=False)
duplicate_groups = exact_text_counts[exact_text_counts > 1]
duplicate_records_mask = text_index["review_text_raw"].isin(duplicate_groups.index)
duplicate_records = text_index[duplicate_records_mask].copy()
assert len(duplicate_groups) == 69
assert len(duplicate_records) == 417
print(f"Exact duplicate-text groups: {len(duplicate_groups):,}")
print(f"Records involved: {len(duplicate_records):,}")
display(duplicate_groups.rename("count").head(50).to_frame())

DUPLICATE_TEXT_TO_INSPECT = ""  # Paste an exact text value to inspect underlying IDs.
if DUPLICATE_TEXT_TO_INSPECT:
    display(duplicate_records[duplicate_records["review_text_raw"].eq(DUPLICATE_TEXT_TO_INSPECT)])

Exact duplicate-text groups: 69
Records involved: 417


,count
review_text_raw,
Ok,69
Tốt,41
Good,39
Quá tệ,27
Tệ,19
105CTOIYEUVN,14
ok,14
Rất tốt,12
Tuyệt vời,11


## Find Review by ID

In [33]:
APP_STORE_REVIEW_ID = ""
GOOGLE_PLAY_REVIEW_ID = ""

if APP_STORE_REVIEW_ID:
    display(app_store_df[app_store_df["review_id"].astype(str).eq(str(APP_STORE_REVIEW_ID))])
else:
    print("Set APP_STORE_REVIEW_ID to inspect an App Store review.")

if GOOGLE_PLAY_REVIEW_ID:
    display(google_play_df[google_play_df["reviewId"].astype(str).eq(str(GOOGLE_PLAY_REVIEW_ID))])
else:
    print("Set GOOGLE_PLAY_REVIEW_ID to inspect a Google Play review.")

Set APP_STORE_REVIEW_ID to inspect an App Store review.
Set GOOGLE_PLAY_REVIEW_ID to inspect a Google Play review.


## Date Distribution Quick View

These simple count tables provide QA context only. They do not select an analysis time range.

In [34]:
app_display_dates = pd.to_datetime(app_store_df["updated_at"], utc=True, errors="coerce")
google_display_dates = pd.to_datetime(google_play_df["at"], utc=True, errors="coerce")
app_year_counts = app_display_dates.dt.year.value_counts().sort_index().rename("App Store")
google_year_counts = google_display_dates.dt.year.value_counts().sort_index().rename("Google Play")
display(pd.concat([app_year_counts, google_year_counts], axis=1).fillna(0).astype(int))

app_month_counts = app_display_dates.dt.strftime("%Y-%m").value_counts().sort_index().rename("App Store")
google_month_counts = google_display_dates.dt.strftime("%Y-%m").value_counts().sort_index().rename("Google Play")
print("Latest 36 observed months")
display(pd.concat([app_month_counts, google_month_counts], axis=1).fillna(0).astype(int).tail(36))

,App Store,Google Play
2023,4,527
2024,221,477
2025,166,209
2026,131,148
2015,0,4
2016,0,3
2017,0,3
2018,0,14
2019,0,45
2020,0,260


Latest 36 observed months


,App Store,Google Play
2021-04,0,27
2021-05,0,52
2021-06,0,73
2021-07,0,39
2021-08,0,42
2021-09,0,24
2021-10,0,26
2021-11,0,550
2021-12,0,118
2022-01,0,34


## Current Processed Outputs

This section references pipeline status only. Raw inspection above continues to use the two primary raw datasets. No processed file is modified.

In [35]:
clean_df = pd.read_csv(CLEAN_PATH, dtype={"review_id": "string"})
rejected_df = pd.read_csv(REJECTED_PATH, dtype={"review_id": "string"})
assert len(clean_df) == 3745
assert len(rejected_df) == 0
processed_status = pd.Series({
    "clean_dataset": str(CLEAN_PATH.relative_to(PROJECT_ROOT)),
    "clean_rows": len(clean_df),
    "rejected_dataset": str(REJECTED_PATH.relative_to(PROJECT_ROOT)),
    "rejected_rows": len(rejected_df),
})
display(processed_status.rename("value").to_frame())

,value
clean_dataset,01_data/processed/reviews_clean_full.csv
clean_rows,3745
rejected_dataset,01_data/processed/reviews_rejected.csv
rejected_rows,0


## Manual QA Notes

- **App Store primary v2:** 522 unique reviews, dated 2023-04-14 through 2026-09-14. It is the broadest validated union of the currently saved public-response evidence. This does **not** mean all lifetime App Store reviews, and public endpoint coverage may be unstable.
- **Google Play:** 3,223 unique reviews, dated 2015-08-13 through 2026-09-21, retrieved through the selected Vietnam collection configurations.
- `collection_locale` is collection metadata, not actual review language. The App Store `lang` field is likewise not assumed to be reliable ground truth.
- Public app reviewers are self-selected and are not representative of all TCInvest customers.
- Very short reviews and exact duplicate text with distinct IDs remain valid records unless independent evidence proves otherwise.
- The current clean full-history output has 3,745 rows and zero rejected rows.
- This notebook performs no cleaning, write-back, sentiment analysis, topic modeling, classification, or time-range selection.